# T6 - Interventions

Understanding the impact of interventions is one of the most common reasons to use a disease model. This tutorial shows how to implement standard interventions, as well as how to define your own custom interventions.  


## Products and interventions

Starsim contains _products_, which can be thought of as the actual test, diagnostic, treatment, or vaccine product being used, as well as _interventions_, which are responsible for delivering the products to the population. 

Depending on what disease you're modeling, you might need to define your own custom products and interventions, or you might be able to directly use some of the examples provided in Starsim.

Starsim includes three basic "types" of products: diagnostics, treatment, and vaccination. There isn't a lot of detail in the templates for each of these, because most of the details about products is specific to a disease. There are also some disease-specific products built in the Starsim's library of diseases - these can generally be found in the `diseases` subfolder (e.g. the cholera interventions are in `cholera.py`. 

Starsim also includes several basic types of intervention:

- `routine_vx()` for routine vaccination, `campaign_vx()` for one-off campaigns 
- similarly, `routine_screening()` and `campaign_screening()` for different types of screening program 
- `treat_num()`, which treats a certain number of people each timestep (by default, as many people as need treatment, but you can also set a maximum).

These are sometimes general enough that they don't need to be tailored to a particular disease, and you can just use them directly. That being said, you are always welcome to tailor them as you like to capture particular features of the intervention you're modeling.

## Vaccination
To create an example, let's first create the parameters that we want to use for the simulation:

In [ ]:
import starsim as ss

pars = dict(
    n_agents = 5_000,
    birth_rate = 20,
    death_rate = 15,
    networks = dict(
        type = 'randomnet', # Or 'random'
        n_contacts = 4
    ),
    diseases = dict(
        type = 'sir',
        dur_inf = 10,
        beta = 0.1,
    )
)

Now we'll create a vaccine product and a vaccination intervention:

In [ ]:
# Create the product - a vaccine with 50% efficacy
my_vaccine = ss.sir_vaccine(efficacy=0.5)

# Create the intervention
vaccine_campaign = ss.routine_vx(
    start_year=2015,    # Begin vaccination in 2015
    prob=0.2,           # 20% coverage
    product=my_vaccine   # Use the MyVaccine product
)

# Now create two sims: a baseline sim and one with the intervention
sim_base = ss.Sim(pars=pars, label='Baseline')
sim_vac  = ss.Sim(pars=pars, interventions=vaccine_campaign, label='With vaccine')

# Run in parallel
msim = ss.parallel(sim_base, sim_vac)

If we want to see the impact, we can create a plot:

In [ ]:
msim.plot('prevalence');

We can see that from the year of introducing the vaccine, prevalence starts to fall. 

## Comparing different interventions

This section illustrates how to create several different types of intervention and compare their impacts.

In [4]:
# Additional imports that will be used later
import numpy as np
import matplotlib.pyplot as plt

# Set up some default disease and network modules
disease = ss.SIR(dur_inf=10, beta=0.1) # SIR model
network = ss.RandomNet(n_contacts=4) # Random network

# Custom function for plotting the results from multiple simulations
def plot_sims(*sims, quantity='prevalence'):
    plt.figure()
    for sim in sims:
        plt.plot(sim.results.timevec, sim.results.sir[quantity], label=sim.label)
    plt.axvline(x=2015, color='k', ls='--')
    plt.title(quantity.title())
    plt.legend()
    plt.ylim(bottom=0)
    plt.show()

### NPI intervention example

Let's consider a "masks" intervention, which has the effect of reducing beta.

In [5]:
class Masks(ss.Intervention):
    def step(self):
        sim = self.sim
        if sim.now > 2015:
            sim.diseases.sir.pars.beta = 0.01 # 10-fold reduction in beta

In [ ]:
sim_masks = ss.Sim(diseases=disease, networks=network, interventions=Masks(), label='Masks')
sim_masks.run()

plot_sims(sim_base, sim_vac, sim_masks)

### Network intervention

Now let's consider an intervention that acts on networks by reducing the number of contacts.

In [7]:
class DecreaseContacts(ss.Intervention):
    def step(self):
        sim = self.sim
        if sim.now > 2015:
            sim.networks.randomnet.pars.n_contacts = 1 # Reduce the number of contacts

In [ ]:
# Now run and plot the simulation with the intervention
sim_contacts = ss.Sim(diseases=disease, networks=network, interventions=DecreaseContacts(), label='Fewer contacts')
sim_contacts.run()

plot_sims(sim_base, sim_vac, sim_masks, sim_contacts)

### Apply all interventions

Finally, let's create a simulation that includes all interventions.

In [ ]:
# Now run and plot the simulation with the intervention
sim_all = ss.Sim(diseases=disease, networks=network, interventions=[vaccine_campaign, Masks(), DecreaseContacts()], label='All interventions')
sim_all.run()

for quantity in ['prevalence', 'new_infections']:
    plot_sims(sim_base, sim_vac, sim_masks, sim_contacts, sim_all, quantity=quantity)

As expected, the more interventions we layer, the lower the number of infections is.